In [ ]:
import sys
sys.path.append('/mnt/scratch/scheduler/proxy_server')
from plot_functions import *
import os

# Default

In [ ]:
import os
import glob
import json
import re
import pandas as pd
import matplotlib.pyplot as plt

def extrair_metricas_json(pasta, metrica="total_e2el_s"):
    padrao_busca = os.path.join(pasta, "baseline_output_fcfs_autellix_rateinf_*progs_run1.json")
    arquivos = glob.glob(padrao_busca)
    
    dados_tabela = []
    
    for arquivo in arquivos:
        nome_arquivo = os.path.basename(arquivo)
        match = re.search(r"rateinf_(\d+)progs_run1\.json", nome_arquivo)
        
        if not match:
            continue
            
        num_programas = int(match.group(1))
        
        with open(arquivo, 'r', encoding='utf-8') as f:
            try:
                conteudo = json.load(f)
                
                # ========================================================
                # NOVO: PROCESSAMENTO DO ARQUIVO CSV (KV CACHE)
                # ========================================================
                # Substitui a extensão para encontrar o CSV correspondente
                arquivo_csv = arquivo.replace(".json", "_kv_cache.csv")
                
                if os.path.exists(arquivo_csv):
                    # Lê o CSV usando pandas
                    df_csv = pd.read_csv(arquivo_csv)
                    
                    # Filtra os valores de kv_cache_percent ignorando os 0.0
                    kv_ativos = df_csv[df_csv['kv_cache_percent'] > 0.0]['kv_cache_percent']
                    running = df_csv[df_csv['running'] > 0.0]['running']
                    waiting = df_csv[df_csv['waiting'] > 0.0]['waiting']
                    
                    # Se houver dados válidos no CSV, calcula e injeta no dicionário 'full_metrics'
                    if True:#not kv_ativos.empty or not running.empty:
                        if "full_metrics" not in conteudo:
                            conteudo["full_metrics"] = {}
                            
                        # Adiciona as métricas calculadas na memória
                        conteudo["full_metrics"]["kv_cache_mean"] = kv_ativos.mean()
                        conteudo["full_metrics"]["kv_cache_median"] = kv_ativos.median()
                        conteudo["full_metrics"]["kv_cache_p75"] = kv_ativos.quantile(0.75)
                        conteudo["full_metrics"]["kv_cache_p90"] = kv_ativos.quantile(0.90)
                        conteudo["full_metrics"]["kv_cache_p95"] = kv_ativos.quantile(0.95)
                        conteudo["full_metrics"]["kv_cache_p99"] = kv_ativos.quantile(0.99)

                        conteudo["full_metrics"]["running_mean"] = running.mean()
                        conteudo["full_metrics"]["running_median"] = running.median()
                        conteudo["full_metrics"]["running_p75"] = running.quantile(0.75)
                        conteudo["full_metrics"]["running_p90"] = running.quantile(0.90)
                        conteudo["full_metrics"]["running_p95"] = running.quantile(0.95)
                        conteudo["full_metrics"]["running_p99"] = running.quantile(0.99)


                        conteudo["full_metrics"]["waiting_mean"] = waiting.mean()
                        conteudo["full_metrics"]["waiting_median"] = waiting.median()
                        conteudo["full_metrics"]["waiting_p75"] = waiting.quantile(0.75)
                        conteudo["full_metrics"]["waiting_p90"] = waiting.quantile(0.90)
                        conteudo["full_metrics"]["waiting_p95"] = waiting.quantile(0.95)
                        conteudo["full_metrics"]["waiting_p99"] = waiting.quantile(0.99)
                # ========================================================
                
                # Lógica especial para o hit_rate do prefix_cache
                if metrica == "prefix_cache_hit_rate":
                    prefix_cache = conteudo.get("full_metrics", {}).get("prefix_cache", {})
                    if prefix_cache:
                        primeiro_engine = list(prefix_cache.keys())[0]
                        valor_metrica = prefix_cache[primeiro_engine].get("hit_rate", None)
                    else:
                        valor_metrica = None
                
                # Lógica padrão para as demais métricas (INCLUINDO AS NOVAS DO KV CACHE)
                else:
                    valor_metrica = conteudo.get("full_metrics", {}).get(metrica, None)
                
                if valor_metrica is not None:
                    dados_tabela.append({
                        "num_programs": num_programas,
                        metrica: valor_metrica
                    })
                else:
                    print(f"Aviso: Métrica '{metrica}' não encontrada para {nome_arquivo}.")
                    
            except Exception as e:
                print(f"Erro ao processar {nome_arquivo}: {e}")
                
    df = pd.DataFrame(dados_tabela)
    if not df.empty:
        df = df.sort_values(by="num_programs").reset_index(drop=True)
        
    return df

def plotar_metricas_json(pasta, metrica="total_e2el_s"):
    df = extrair_metricas_json(pasta, metrica)
    
    if df.empty:
        print(f"Nenhum dado encontrado na pasta '{pasta}' para a métrica '{metrica}'.")
        return
    
    plt.figure(figsize=(10, 6))
    plt.scatter(df["num_programs"], df[metrica], color="#1f77b4", marker="o", s=100, zorder=3)
    
    plt.title(f"Impacto do Número de Programas na Métrica: {metrica}", fontsize=14, pad=15)
    plt.xlabel("Número de Programas (num_programs)", fontsize=12)
    plt.ylabel(metrica, fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7, zorder=0)
    
    plt.xlim(df["num_programs"].min() - 1, df["num_programs"].max() + 1)
    
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import glob
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  # Necessário para a regressão linear

# (Mantenha a função extrair_metricas_json que criamos no passo anterior)

def plotar_regressao(pasta, metrica="total_e2el_s"):
    """
    Lê arquivos JSON, exibe um gráfico de pontos e plota uma regressão linear 
    calculada com pontos < 350, mas estendida por todo o eixo X do gráfico.
    """
    df = extrair_metricas_json(pasta, metrica)
    
    if df.empty:
        print(f"Nenhum dado encontrado na pasta '{pasta}' para a métrica '{metrica}'.")
        return
    
    plt.figure(figsize=(10, 6))
    
    # Plota todos os pontos (dispersão geral)
    plt.scatter(df["num_programs"], df[metrica], color="#1f77b4", marker="o", s=100, zorder=3, label="Métricas executadas")
    
    # Filtra os dados para calcular a regressão APENAS com num_programs < 350
    df_filtrado = df[df["num_programs"] < 350]
    
    if len(df_filtrado) > 1:
        x_treino = df_filtrado["num_programs"]
        y_treino = df_filtrado[metrica]
        
        # Calcula a regressão linear (polinômio de grau 1) com os dados filtrados
        coeficientes = np.polyfit(x_treino, y_treino, 1)
        polinomio = np.poly1d(coeficientes)
        
        # GERA A LINHA PARA TODA A EXTENSÃO: Usa o min e max do DataFrame original (df)
        x_linha = np.linspace(df["num_programs"].min(), df["num_programs"].max(), 100)
        y_linha = polinomio(x_linha)
        
        # Plota a linha vermelha tracejada projetada
        plt.plot(x_linha, y_linha, color="red", linestyle="--", linewidth=2.5, zorder=4,
                 label=f"Projeção da Tendência (<350)\ny = {coeficientes[0]:.4g}x + {coeficientes[1]:.4g}")
    elif len(df_filtrado) == 1:
        print("Aviso: Apenas 1 ponto encontrado abaixo de 350 programas. Regressão linear impossível.")
    else:
        print("Aviso: Nenhum ponto encontrado abaixo de 350 programas.")

    # Estilização
    plt.title(f"Impacto do Número de Programas na Métrica: {metrica}", fontsize=14, pad=15)
    plt.xlabel("Número de Programas (num_programs)", fontsize=12)
    plt.ylabel(metrica, fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7, zorder=0)
    plt.legend(fontsize=10)
    
    plt.xlim(df["num_programs"].min() - 1, df["num_programs"].max() + 1)
    
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import glob
import json

def recalcular_throughput_nos_jsons(pasta):
    """
    Lê os arquivos JSON, recalcula os throughputs de tokens (input, output e total)
    baseado no tempo 'total_e2el_s' e salva a alteração diretamente no próprio arquivo.
    """
    padrao_busca = os.path.join(pasta, "baseline_output_fcfs_autellix_rateinf_*progs_run1.json")
    arquivos = glob.glob(padrao_busca)
    
    arquivos_atualizados = 0
    
    for arquivo in arquivos:
        nome_arquivo = os.path.basename(arquivo)
        
        # 1. Lê o conteúdo atual do JSON
        with open(arquivo, 'r', encoding='utf-8') as f:
            try:
                conteudo = json.load(f)
            except json.JSONDecodeError:
                print(f"Erro ao ler o arquivo {nome_arquivo}. JSON inválido.")
                continue
                
        # 2. Verifica se a chave 'full_metrics' existe
        if "full_metrics" in conteudo:
            fm = conteudo["full_metrics"]
            chaves_necessarias = ["total_input_tokens", "total_output_tokens", "total_e2el_s"]
            
            # Garante que os dados necessários para o cálculo estão presentes
            if all(k in fm for k in chaves_necessarias):
                in_tokens = fm["total_input_tokens"]
                out_tokens = fm["total_output_tokens"]
                tempo_total = fm["total_e2el_s"]
                
                # Previne erro de divisão por zero
                if tempo_total > 0:
                    # Faz os recálculos
                    in_throughput = in_tokens / tempo_total
                    out_throughput = out_tokens / tempo_total
                    total_throughput = (in_tokens + out_tokens) / tempo_total
                    
                    # Substitui/Adiciona os valores no dicionário
                    conteudo["full_metrics"]["input_token_throughput"] = in_throughput
                    conteudo["full_metrics"]["output_token_throughput"] = out_throughput
                    conteudo["full_metrics"]["total_token_throughput"] = total_throughput
                    conteudo["full_metrics"]["total_token"] = in_tokens+out_tokens
                    
                    # 3. Salva as alterações de volta no arquivo original
                    with open(arquivo, 'w', encoding='utf-8') as f_out:
                        # indent=2 mantém o arquivo legível e formatado bonitinho
                        json.dump(conteudo, f_out, indent=2, ensure_ascii=False)
                        
                    print(f"Atualizado: {nome_arquivo}")
                    arquivos_atualizados += 1
                else:
                    print(f"Ignorado: {nome_arquivo} (total_e2el_s é zero ou negativo)")
            else:
                print(f"Ignorado: {nome_arquivo} (faltam chaves base para o cálculo)")
        else:
            print(f"Ignorado: {nome_arquivo} (sem chave 'full_metrics')")
            
    print(f"\nProcesso concluído! {arquivos_atualizados} arquivos foram atualizados.")

In [ ]:
plot_metrics(os.path.join("FCFS_608_512", f"baseline_output_fcfs_autellix_rateinf_608progs_run1_kv_cache.csv"))

In [ ]:
plot_metrics(os.path.join("FCFS_608_512", f"baseline_output_fcfs_autellix_rateinf_608progs_run1_kv_cache.csv"))

In [ ]:
plotar_metricas_json("FCFS_608_512", metrica="waiting_mean")

In [ ]:
plotar_metricas_json("FCFS_608_512", metrica="running_p90")

In [ ]:
plotar_metricas_json("FCFS_608_512", metrica="running_p99")

In [ ]:
plotar_metricas_json("FCFS_608_512", metrica="running_p90")

In [ ]:
plotar_metricas_json("FCFS_608_512", metrica="running_mean")

In [ ]:
plotar_metricas_json("FCFS_608_512", metrica="running_p75")

In [ ]:
recalcular_throughput_nos_jsons("FCFS_608_512")

In [ ]:
tabela_e2el = extrair_metricas_json("running_mean")
print(tabela_e2el)


In [ ]:
# tabela_tokens = extrair_metricas_json("var_50it", metrica="mean_e2el_per_program")
tabela_tokens = extrair_metricas_json("FCFS_608_512", metrica="prefix_cache_hit_rate")
print(tabela_tokens)

In [ ]:
plotar_metricas_json("FCFS_608_512", metrica="total_token")
plotar_metricas_json("FCFS_608_512", metrica="total_input_tokens")
plotar_metricas_json("FCFS_608_512", metrica="total_output_tokens")

In [ ]:
# Exemplo 1: Gerar gráfico para a métrica padrão (total_e2el_s)
# plotar_regressao("var_50it_mem", metrica="total_e2el_s")
# plotar_regressao("var_50it", metrica="total_e2el_s")


plotar_metricas_json("var_50it_mem", metrica="total_token_throughput")
plotar_metricas_json("var_50it_mem", metrica="total_output_tokens")
plotar_metricas_json("var_50it_mem", metrica="output_token_throughput")
plotar_metricas_json("var_50it_mem", metrica="total_e2el_s")
plotar_metricas_json("var_50it_mem", metrica="kv_cache_p75")

plotar_metricas_json("var_50it_mem", metrica="prefix_cache_hit_rate")

# plotar_regressao("var_50it_mem", metrica="total_e2el_s")


In [ ]:
PATH = "38prog"
RUN = "1"
RATE="inf"
METRIC = "total_e2el_s"

In [ ]:
limpar_logs(PATH)

In [ ]:
merge_engine_request_counts(PATH)

## Resultados e hit rate

In [ ]:
results(PATH, METRIC)

In [ ]:
results(PATH, METRIC, 2)

In [ ]:
results(PATH, METRIC, 4)

In [ ]:
results(PATH, METRIC, 8)

In [ ]:
results_hit_rate(PATH)

In [ ]:
results(PATH, "total_wiki_search_time_s")

## Load Balancers

### Autelix

In [ ]:
plot_metrics(os.path.join(PATH, f"baseline_output_fcfs_autellix_rateinf_run1_kv_cache.csv"))

In [ ]:
plot_engine_metrics(os.path.join(PATH, f"baseline_output_atlas_service_cumulative_autellix_rate{RATE}_run{RUN}.json"))

In [ ]:
#analisar_preferred_engines("new_outputs_lats/nocache_def_llama_100prog_50it_5gen_30parall_rate01/processes_summary_atlas_service_cumulative_ordered-dynamic-autellix-reorder_rate0.1_run1.json")
analisar_preferred_engines(os.path.join(PATH, f"baseline_processes_summary_atlas_service_cumulative_autellix_rate{RATE}_run{RUN}.json"))


### Least Load

In [ ]:
plot_metrics(os.path.join(PATH, f"output_atlas_service_cumulative_least-total-load_rate{RATE}_run{RUN}_kv_cache.csv"))

In [ ]:
analisar_preferred_engines(os.path.join(PATH, f"processes_summary_atlas_service_cumulative_least-total-load_rate{RATE}_run{RUN}.json"))

In [ ]:
plot_engine_metrics(os.path.join(PATH, f"output_atlas_service_cumulative_least-total-load_rate{RATE}_run{RUN}.json"))

### Ordered

In [ ]:
plot_metrics(os.path.join(PATH, f"output_atlas_service_cumulative_ordered-dynamic-autellix_rate{RATE}_run{RUN}_kv_cache.csv"))

In [ ]:
plot_engine_metrics(os.path.join(PATH, f"output_atlas_service_cumulative_ordered-dynamic-autellix_rate{RATE}_run{RUN}.json"))

In [ ]:
analisar_preferred_engines(os.path.join(PATH, f"processes_summary_atlas_service_cumulative_ordered-dynamic-autellix_rate{RATE}_run{RUN}.json"))

In [ ]:
plot_engine_metrics(os.path.join(PATH, f"output_atlas_service_cumulative_ordered-dynamic-autellix_rate{RATE}_run{RUN}.json"))

### Probabilistic Cascade

In [ ]:
plot_metrics(os.path.join(PATH, f"output_atlas_service_cumulative_probabilistic-cascade-autellix_rate{RATE}_run1_kv_cache.csv"))

In [ ]:
analisar_preferred_engines(os.path.join(PATH, f"processes_summary_atlas_service_cumulative_probabilistic-cascade-autellix_rate{RATE}_run{RUN}.json"))